# Module 02 — Write Problems (Colab)

NovaBridge's agent charges a client's quarterly advisory fee. Agents **retry** —
a timeout, a transient error — so the same charge can be attempted more than once.
The fee must reach the client's card **exactly once**.

You'll change one file, `your_fix.py`, and meet two traps on the way:

1. Leaning on the database's unique key dedupes your *row* — but not the *card*.
2. Keying the charge on the agent's *memo* looks fine on a replay — until the model
   rewrites the memo on the next attempt.

## 1. Set up (Postgres + recorded model outputs, ~2 min)

In [ ]:
%cd /content
!rm -rf repo
!git clone https://github.com/sanbhaumik/workshop-designing-data-infra-for-ai-agents.git repo
%cd repo

In [ ]:
!SKIP_OLLAMA=1 bash setup.sh
# Failsafe: ensure Python deps are installed even on PEP 668 runtimes.
!python -m pip install --break-system-packages -q "psycopg[binary]==3.2.9"

In [ ]:
import os
os.environ['NOVA_LLM'] = 'frozen'  # replay recorded real model outputs (deterministic)
os.environ['DATABASE_URL'] = 'postgresql://postgres@localhost:5432/nova'

## 2. Meet the agent

The "agent" is just code you can call: it reads a billing note and writes a one-line
payment memo. Here's the note it reads:

In [ ]:
import sys, importlib, hashlib
sys.path.insert(0, 'modules/02_write_path')

from nova.agent import load_document, payment_memo_prompt
from nova.effects import PaymentGateway
from nova.llm import get_llm
from nova.store import get_store
import your_fix

llm = get_llm()
store = get_store(); store.init_schema()
doc = load_document('alpha', 'billing_instruction.md')
print(doc)

The agent reads that note and writes a payment memo. Run it once:

In [ ]:
memo_1 = llm.complete(payment_memo_prompt('alpha', 'Q1-2026', doc, 1))
print('attempt 1:', memo_1)

Now the agent **retries** — same fee, same inputs. Watch the memo it writes this time:

In [ ]:
memo_2 = llm.complete(payment_memo_prompt('alpha', 'Q1-2026', doc, 2))
print('attempt 2:', memo_2)
print('same memo?', memo_1 == memo_2)

Different words, same fee. Hold on to that — it matters later. The charge itself goes
through a **payment gateway**: an external processor where every call moves real money
and nothing is undone. Your own `charges` table has a unique key on the fee.

## 3. The obvious fix: lean on the table's unique key

Your `charges` table already has a unique key, so a duplicate insert is ignored. The
natural move: charge the card, then record the row — let the table dedup.

### ✋ Predict #1

We run the fee once, then retry it. After both attempts: **how many rows land in the
`charges` table, and how many times is the card charged?**

In [ ]:
def table_key_charge(gateway, store, client_id, period, amount, memo):
    key = hashlib.sha256(f'{client_id}|{period}'.encode()).hexdigest()
    gateway.charge(client_id, amount, memo)      # irreversible effect fires first
    store.record_charge(key, client_id, amount)  # unique key -> only one row survives

store.reset_demo(); gateway = PaymentGateway()
for memo in (memo_1, memo_2):   # the agent runs, then retries
    table_key_charge(gateway, store, 'alpha', 'Q1-2026', 2500, memo)

print('rows in charges table:', len(store.get_charges('alpha')))
print('times the card was charged:', len(gateway.charges_for('alpha')))

See it in the **real database** — one clean row:

In [ ]:
!psql "$DATABASE_URL" -c "SELECT client_id, amount FROM charges;"

### The aha (1 of 2)

The table looks perfect: **one** row. But the gateway charged the client **twice** —
the money moved before the record deduped anything. A unique key protects *your records,
not the client's card*. The guard has to sit **before** the irreversible effect.

## 4. Guard the effect — first attempt

Move the check ahead of the charge: if this fee was already charged, don't touch the
gateway. Here's the file you'll edit:

In [ ]:
print(open('modules/02_write_path/your_fix.py').read())

Edit the cell below. Key the charge on the agent's memo, and check `already_charged`
**before** charging:

In [ ]:
%%writefile modules/02_write_path/your_fix.py
import hashlib


def charge_key(client_id, billing_period, memo):
    return hashlib.sha256(memo.encode('utf-8')).hexdigest()  # key on the memo


def charge_client_fee(gateway, store, client_id, billing_period, amount, memo):
    key = charge_key(client_id, billing_period, memo)
    if store.already_charged(key):
        return                                # guard BEFORE the effect
    gateway.charge(client_id, amount, memo)
    store.record_charge(key, client_id, amount)

Run the retry the easy way — the **same memo replayed** — and check the card:

In [ ]:
importlib.reload(your_fix)
store.reset_demo(); gateway = PaymentGateway()
for _ in range(2):   # the retry replays the SAME memo
    your_fix.charge_client_fee(gateway, store, 'alpha', 'Q1-2026', 2500, memo_1)

print('times the card was charged:', len(gateway.charges_for('alpha')))
!python -m pytest modules/02_write_path/test_write.py::test_replayed_retry_charges_the_client_only_once -q

Charged once, test green. Looks fixed. **Is it?**

### ✋ Predict #2

A real retry isn't always a clean replay — the agent re-runs and the model **rewrites
the memo** (that's `memo_1` vs `memo_2` from the top). Same fee, same client, same
period, different memo. With the charge keyed on the memo, **how many times is the card
charged now?**

In [ ]:
store.reset_demo(); gateway = PaymentGateway()
for memo in (memo_1, memo_2):   # the agent re-ran; the model rewrote the memo
    your_fix.charge_client_fee(gateway, store, 'alpha', 'Q1-2026', 2500, memo)

print('times the card was charged:', len(gateway.charges_for('alpha')))

In [ ]:
!python -m pytest modules/02_write_path/test_write.py -q

### The aha (2 of 2)

The double charge came back. The memo is the agent's **output**, and the model rewrites
it every run — so the "same" fee got two different keys and slipped past the guard. An
idempotency key must be the one thing that *doesn't* change across attempts: the fee's
**intent** — which client, which period.

In [ ]:
%%writefile modules/02_write_path/your_fix.py
import hashlib


def charge_key(client_id, billing_period, memo):
    # key on INTENT (client + period); ignore the memo, it changes every run
    return hashlib.sha256(f'{client_id}|{billing_period}'.encode('utf-8')).hexdigest()


def charge_client_fee(gateway, store, client_id, billing_period, amount, memo):
    key = charge_key(client_id, billing_period, memo)
    if store.already_charged(key):
        return
    gateway.charge(client_id, amount, memo)
    store.record_charge(key, client_id, amount)

In [ ]:
importlib.reload(your_fix)
!python -m pytest modules/02_write_path/test_write.py -q

### The aha

Exactly-once against an irreversible effect needs **both**: guard *before* the effect,
and key on the *intent* of the work, never on the model's output. Run the full
before/after — both retry kinds at once:

In [ ]:
!python modules/02_write_path/compare.py

### Optional: run the real local model

Everything above replayed recorded outputs so the double charge is reproducible. To watch
the same design play out against a live local model, install Ollama and set `NOVA_LLM=ollama`
(see `SETUP.md`). The design lesson is identical — only the memo wording changes.